# Modelo de recomendación de cultivos con datos reales de Europa

Sustituye a `entrenar_modelo.ipynb`. Aquel dataset de Kaggle no servía para recomendar: en él el cultivo no depende de las condiciones (clasificarlo con todas sus variables acierta lo mismo que el azar), así que el modelo ordenaba siempre por t/ha y salía Sugarcane.

**Datos (todos reales, se descargan aquí, sin claves):**
- **Eurostat `apro_cpshr`**: superficie, producción y rendimiento (t/ha) de cada cultivo por región NUTS2 y año, 2010–2024.
- **NASA POWER**: clima mensual de cada región y año (temperatura, lluvia, radiación, humedad relativa).
- **SoilGrids (ISRIC)**: pH del suelo (primeros 30 cm) de cada región, del mapa mundial a 5 km.
- **GISCO**: punto interior de cada región, para pedir su clima y su suelo.

**Variables**: temperatura y lluvia de cada estación del año, radiación de primavera y verano, humedad relativa media y pH del suelo.

**Dos modelos** (gradient boosting):
1. `cuota`: qué parte de la tierra dedican a cada cultivo las regiones con ese clima y ese suelo. Es lo que hacen de verdad los agricultores, que ya cuentan con costes, rotación y mercado.
2. `rendimiento`: t/ha esperadas (solo con el clima: el pH no mejora esta predicción).

El backend calcula las mismas variables con las balizas (temperatura, humedad y pH) y Meteomatics y ordena por `cuota × ventaja`, donde la ventaja es lo que el cultivo rinde y se paga en esa finca frente a lo habitual en Europa (precio de venta de Eurostat). No se ordena por ingreso bruto por hectárea: sin costes, la patata (que factura diez veces más que un cereal) saldría siempre primera.

Ejecuta todas las celdas (Entorno de ejecución → Ejecutar todas). La descarga tarda unos minutos la primera vez.

In [ ]:
import json, os, subprocess, sys, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor

import joblib
import numpy as np
import pandas as pd
import sklearn

try:
    import rasterio
except ImportError:   # Colab no lo trae: hace falta para leer el mapa de pH
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
    import rasterio
from rasterio.warp import transform as reproyectar
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict

print("scikit-learn", sklearn.__version__)

### 1. Descarga y construcción del conjunto de datos
Cada fila es un cultivo en una región y un año: su superficie, su rendimiento, el clima de ese año y el pH del suelo de la región. Las descargas se guardan en `cache/` para no repetirlas.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
GISCO = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_LB_{}_4326_LEVL_2.geojson"
POWER = "https://power.larc.nasa.gov/api/temporal/monthly/point"
SOILGRIDS = "https://files.isric.org/soilgrids/latest/data_aggregated/5000m/phh2o/phh2o_{}_mean_5000.tif"
# Capas de SoilGrids para el pH de los primeros 30 cm (peso = grosor de la capa en cm).
CAPAS_PH = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}
RADIO_PH_GRADOS = 0.3      # el pH de la región es la media de una rejilla de puntos alrededor de su punto interior
PRIMER_ANIO, ULTIMO_ANIO = 2010, 2024
CACHE = "cache"

# código de cultivo de Eurostat (apro_cpshr) -> (id en la tabla `cultivos`, nombre)
CULTIVOS = {
    "C1110": (1, "Trigo blando de invierno"), "C1120": (2, "Trigo duro"), "C1300": (3, "Cebada"),
    "C1410": (4, "Avena"), "C1500": (5, "Maíz grano"), "C1700": (7, "Sorgo grano"), "C2000": (8, "Arroz"),
    "C1200": (58, "Centeno"), "C1600": (59, "Triticale"), "R1000": (20, "Patata"),
    "R2000": (19, "Remolacha azucarera"), "I1110": (16, "Colza"), "I1120": (15, "Girasol"),
    "I1130": (65, "Soja"), "I3000": (68, "Tabaco"), "P1200": (11, "Haba seca"),
    "I1140": (67, "Lino"), "G3000": (61, "Maíz forrajero (ensilado)"),
}
AREA_MINIMA_KHA = 0.5      # un cultivo "se da" en una región-año si ocupa al menos 500 ha
MIN_CULTIVOS_REGION = 5    # región-año que informa de al menos estos cultivos (para los ceros)


def _json(url, intentos=4):
    for i in range(intentos):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return json.loads(r.read().decode())
        except Exception:
            if i == intentos - 1:
                raise
            time.sleep(3 * (i + 1))


def _en_cache(nombre, fn):
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if os.path.exists(ruta):
        return json.load(open(ruta))
    datos = fn()
    json.dump(datos, open(ruta, "w"))
    return datos


def _filas(d):
    ids, sizes = d["id"], d["size"]
    codes = [sorted(d["dimension"][k]["category"]["index"], key=d["dimension"][k]["category"]["index"].get) for k in ids]
    for k, v in d["value"].items():
        k, pos = int(k), []
        for s in reversed(sizes):
            k, i = divmod(k, s)
            pos.append(i)
        pos.reverse()
        yield {dim: codes[j][i] for j, (dim, i) in enumerate(zip(ids, pos))}, v


def rendimientos_regionales():
    """Eurostat apro_cpshr -> DataFrame región, año, cultivo, area_kha, rendimiento_t_ha (None si no hay)."""
    bruto = {}
    cultivos = "&".join(f"crops={c}" for c in CULTIVOS)
    for medida in ("YLD_HUMD_EU_T_HA", "AR_THS_HA", "HPRD_HUMD_EU_THS_T"):
        url = f"{EUROSTAT}/apro_cpshr?format=JSON&lang=EN&strucpro={medida}&{cultivos}&sinceTimePeriod={PRIMER_ANIO}&untilTimePeriod={ULTIMO_ANIO}"
        d = _en_cache(f"cpshr_{medida}.json", lambda: _json(url))
        for fila, valor in _filas(d):
            if len(fila["geo"]) == 4 and not fila["geo"].endswith("ZZ"):   # solo regiones NUTS2
                bruto.setdefault((fila["geo"], int(fila["time"]), fila["crops"]), {})[medida] = valor
    filas = []
    for (region, anio, cultivo), v in bruto.items():
        area, prod = v.get("AR_THS_HA"), v.get("HPRD_HUMD_EU_THS_T")
        rend = v.get("YLD_HUMD_EU_T_HA") or (prod / area if area and prod else None)
        filas.append({"region": region, "anio": anio, "cultivo": cultivo, "area_kha": area or 0.0,
                      "rendimiento_t_ha": rend if rend and rend > 0 else None})
    return pd.DataFrame(filas)


def centroides():
    """Punto interior de cada región NUTS2 (GISCO; varias versiones de NUTS porque los códigos cambian)."""
    puntos = {}
    for version in (2016, 2021, 2024):   # la más nueva pisa a las anteriores
        d = _en_cache(f"nuts_{version}.json", lambda: _json(GISCO.format(version)))
        for f in d["features"]:
            lon, lat = f["geometry"]["coordinates"]
            puntos[f["properties"]["NUTS_ID"]] = (round(lat, 3), round(lon, 3))
    return puntos


def clima_mensual(lat, lon):
    """NASA POWER: {(año, mes): {t, lluvia_mm_dia, rad_mj, hr}} del punto."""
    q = urllib.parse.urlencode({"parameters": "T2M,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M", "community": "AG",
                                "longitude": lon, "latitude": lat, "start": PRIMER_ANIO, "end": ULTIMO_ANIO, "format": "JSON"})
    p = _en_cache(f"power_{lat}_{lon}.json", lambda: _json(f"{POWER}?{q}")["properties"]["parameter"])
    claves = {"T2M": "t", "PRECTOTCORR": "lluvia_mm_dia", "ALLSKY_SFC_SW_DWN": "rad_mj", "RH2M": "hr"}
    meses = {}
    for param, clave in claves.items():
        for fecha, valor in p[param].items():
            anio, mes = int(fecha[:4]), int(fecha[4:])
            if 1 <= mes <= 12 and valor is not None and valor > -900:
                meses.setdefault((anio, mes), {})[clave] = valor
    return meses


def ph_del_suelo(puntos):
    """SoilGrids (ISRIC): pH en agua de los primeros 30 cm alrededor de cada punto {región: (lat, lon)} -> {región: pH}."""
    os.makedirs(CACHE, exist_ok=True)
    desplazamientos = np.linspace(-RADIO_PH_GRADOS, RADIO_PH_GRADOS, 5)
    suma = {r: 0.0 for r in puntos}
    peso = {r: 0.0 for r in puntos}
    for capa, grosor in CAPAS_PH.items():
        ruta = os.path.join(CACHE, f"ph_{capa}.tif")
        if not os.path.exists(ruta):
            urllib.request.urlretrieve(SOILGRIDS.format(capa), ruta)
        with rasterio.open(ruta) as mapa:
            for region, (lat, lon) in puntos.items():
                lats = [lat + d for d in desplazamientos for _ in desplazamientos]
                lons = [lon + d for _ in desplazamientos for d in desplazamientos]
                xs, ys = reproyectar("EPSG:4326", mapa.crs, lons, lats)
                valores = [v[0] for v in mapa.sample(zip(xs, ys))]
                # El mapa guarda pH × 10; 0, negativos y el valor "sin dato" (mar, ciudad) se descartan.
                validos = [v / 10 for v in valores if v != mapa.nodata and 20 < v < 110]
                if validos:
                    suma[region] += grosor * sum(validos) / len(validos)
                    peso[region] += grosor
    return {r: suma[r] / peso[r] for r in puntos if peso[r] > 0}


DIAS_MES = [31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
# Meses (1-12) de cada estación. El invierno es enero, febrero y diciembre del mismo año.
ESTACIONES = {"invierno": [12, 1, 2], "primavera": [3, 4, 5], "verano": [6, 7, 8], "otono": [9, 10, 11]}
VARIABLES_CLIMA = ([f"t_{e}" for e in ESTACIONES] + [f"lluvia_{e}_mm" for e in ESTACIONES]
                   + ["radiacion_primavera_mj_dia", "radiacion_verano_mj_dia", "hr_media"])
VARIABLES = VARIABLES_CLIMA + ["ph_suelo"]


def variables_del_anio(meses):
    """Las variables de clima del modelo a partir de 12 meses {mes: {t, lluvia_mm_dia, rad_mj, hr}}. None si falta algo.
    El backend calcula las mismas con los datos de las balizas y Meteomatics."""
    if any(m not in meses or len(meses[m]) < 4 for m in range(1, 13)):
        return None
    v = {}
    for estacion, ms in ESTACIONES.items():
        v[f"t_{estacion}"] = sum(meses[m]["t"] for m in ms) / 3
        v[f"lluvia_{estacion}_mm"] = sum(meses[m]["lluvia_mm_dia"] * DIAS_MES[m - 1] for m in ms)
    for estacion in ("primavera", "verano"):
        v[f"radiacion_{estacion}_mj_dia"] = sum(meses[m]["rad_mj"] for m in ESTACIONES[estacion]) / 3
    v["hr_media"] = sum(meses[m]["hr"] for m in range(1, 13)) / 12
    return v


def construir():
    rend = rendimientos_regionales()
    puntos = centroides()
    regiones = sorted(set(rend.region) & set(puntos))
    print(f"regiones con datos: {rend.region.nunique()}, con centroide: {len(regiones)}")
    with ThreadPoolExecutor(4) as pool:
        climas = dict(zip(regiones, pool.map(lambda r: clima_mensual(*puntos[r]), regiones)))
    filas = []
    for region in regiones:
        for anio in range(PRIMER_ANIO, ULTIMO_ANIO + 1):
            v = variables_del_anio({m: climas[region].get((anio, m), {}) for m in range(1, 13)})
            if v:
                filas.append({"region": region, "anio": anio, "latitud": puntos[region][0], "longitud": puntos[region][1], **v})
    clima = pd.DataFrame(filas)
    ph = ph_del_suelo({r: puntos[r] for r in regiones})
    clima["ph_suelo"] = clima.region.map(ph)      # NaN si el mapa no tiene dato: el modelo admite huecos

    # Presencia: en cada región-año que informa de bastantes cultivos, cada cultivo de la lista se da (1) o no (0).
    informa = rend[rend.area_kha > 0].groupby(["region", "anio"]).cultivo.nunique()
    validas = informa[informa >= MIN_CULTIVOS_REGION].reset_index()[["region", "anio"]]
    rejilla = validas.merge(pd.DataFrame({"cultivo": list(CULTIVOS)}), how="cross")
    datos = rejilla.merge(rend, on=["region", "anio", "cultivo"], how="left")
    datos["area_kha"] = datos.area_kha.fillna(0.0)
    datos["se_cultiva"] = (datos.area_kha >= AREA_MINIMA_KHA).astype(int)
    # Rendimientos imposibles (errores de unidades en origen): fuera los que pasan de 4 veces la mediana del cultivo.
    mediana = datos.groupby("cultivo").rendimiento_t_ha.transform("median")
    datos.loc[(datos.rendimiento_t_ha > 4 * mediana) | (datos.se_cultiva == 0), "rendimiento_t_ha"] = None
    datos = datos.merge(clima, on=["region", "anio"])
    datos["cultivo_id"] = datos.cultivo.map(lambda c: CULTIVOS[c][0])
    datos["cultivo_nombre"] = datos.cultivo.map(lambda c: CULTIVOS[c][1])
    return datos

In [ ]:
datos = construir()
datos.to_csv("entrenamiento_eurostat_power.csv.gz", index=False, float_format="%.4g",
             columns=[c for c in datos.columns if c not in ("cultivo_nombre", "cultivo_id")])
print(datos.shape, "| regiones:", datos.region.nunique(), "| países:", datos.region.str[:2].nunique(),
      "| años:", datos.anio.min(), "-", datos.anio.max())
print("filas con rendimiento:", datos.rendimiento_t_ha.notna().sum())
print(datos.groupby("cultivo_nombre").agg(filas_con_rendimiento=("rendimiento_t_ha", "count"), se_cultiva=("se_cultiva", "mean"),
                                         rendimiento_mediano=("rendimiento_t_ha", "median")).round(2).to_string())

### 2. Evaluación
Validación cruzada dejando **regiones enteras fuera**: mide si el modelo acierta en sitios que no ha visto, que es como se usa (una finca nueva). El rendimiento se predice relativo a la mediana de cada cultivo para que la patata (30 t/ha) no pese más que el trigo (5 t/ha); un R² de 0 sería decir siempre la mediana.

In [ ]:
codigos = sorted(CULTIVOS)                      # la posición es el código numérico del cultivo
datos["cultivo_cod"] = datos.cultivo.map(codigos.index)
COLUMNAS = ["cultivo_cod"] + VARIABLES                    # cuota: clima + pH del suelo
COLUMNAS_REND = ["cultivo_cod"] + VARIABLES_CLIMA         # rendimiento: solo clima
cv = GroupKFold(n_splits=5)

def regresor():
    return HistGradientBoostingRegressor(categorical_features=[0], max_iter=400, learning_rate=0.06, random_state=0)

# Cuota de superficie de cada cultivo en su región y año (suman 1 entre los cultivos del modelo)
datos["cuota"] = datos.area_kha / datos.groupby(["region", "anio"]).area_kha.transform("sum")
X = datos[COLUMNAS]
datos["cuota_pred"] = cross_val_predict(regresor(), X, datos.cuota, cv=cv, groups=datos.region)
r2_cuota = r2_score(datos.cuota, datos.cuota_pred)
grupos = datos.groupby(["region", "anio"])
orden = grupos.apply(lambda g: g.cuota.corr(g.cuota_pred, method="spearman")).mean()
principal = grupos.apply(lambda g: g.loc[g.cuota.idxmax(), "cultivo"] in set(g.nlargest(2, "cuota_pred").cultivo)).mean()
print(f"CUOTA: R² {r2_cuota:.3f} en regiones no vistas")
print(f"  orden de los cultivos dentro de cada región y año: correlación de rangos media {orden:.2f}")
print(f"  el cultivo principal real está entre los 2 primeros del modelo en el {100 * principal:.0f} % de los casos")

# Rendimiento
r = datos[datos.rendimiento_t_ha.notna()].copy()
mediana = r.groupby("cultivo").rendimiento_t_ha.median()
r["rel"] = r.rendimiento_t_ha / r.cultivo.map(mediana)
pr = cross_val_predict(regresor(), r[COLUMNAS_REND], r.rel, cv=cv, groups=r.region)
r["pred"] = pr * r.cultivo.map(mediana)
r2 = r2_score(r.rel, pr)
print(f"RENDIMIENTO: R² {r2:.3f} en regiones no vistas (0 = decir siempre la mediana del cultivo)")

filas = []
for c in codigos:
    g, sel = r[r.cultivo == c], datos[datos.cultivo == c]
    filas.append({"cultivo": CULTIVOS[c][1], "cuota media %": round(100 * sel.cuota.mean(), 1),
                  "error medio cuota %": round(100 * mean_absolute_error(sel.cuota, sel.cuota_pred), 1), "filas": len(g),
                  "mediana t/ha": round(mediana[c], 2), "R² rendimiento": round(r2_score(g.rendimiento_t_ha, g.pred), 2),
                  "error medio t/ha": round(mean_absolute_error(g.rendimiento_t_ha, g.pred), 2),
                  "error diciendo la mediana": round(mean_absolute_error(g.rendimiento_t_ha, np.full(len(g), mediana[c])), 2)})
print(pd.DataFrame(filas).to_string(index=False))

### 3. Modelo final y exportación
Se entrena con todos los datos y se guarda en un solo archivo. Cópialo, junto con el conjunto de datos, al backend:

- `modelo_cultivos_europa.joblib` → `modelo_predicciones/`
- `entrenamiento_eurostat_power.csv.gz` → `modelo_predicciones/datos/`

Si el backend tiene otra versión de scikit-learn y el `.joblib` no carga, el backend entrena solo al arrancar con ese mismo conjunto de datos y estos mismos parámetros (`modelo_predicciones/modelo_europa.py`).

In [ ]:
m_cuota = regresor().fit(X, datos.cuota)
m_rendimiento = regresor().fit(r[COLUMNAS_REND], r.rel)
meta = {
    "columnas_cuota": COLUMNAS, "columnas_rendimiento": COLUMNAS_REND, "variables": VARIABLES, "codigos": codigos,
    "cultivos": {c: {"id": CULTIVOS[c][0], "nombre": CULTIVOS[c][1], "mediana_t_ha": float(mediana[c])} for c in codigos},
    "rangos_entrenamiento": {v: [float(datos[v].quantile(0.01)), float(datos[v].quantile(0.99))] for v in VARIABLES},
    "sklearn": sklearn.__version__,
    "evaluacion": {"r2_cuota": float(r2_cuota), "cultivo_principal_en_top2": float(principal), "r2_rendimiento_relativo": float(r2)},
    "datos": "Eurostat apro_cpshr 2010-2024 (NUTS2) + NASA POWER mensual + SoilGrids pH",
}
joblib.dump({"cuota": m_cuota, "rendimiento": m_rendimiento, "meta": meta}, "modelo_cultivos_europa.joblib", compress=3)
print("guardado modelo_cultivos_europa.joblib,", os.path.getsize("modelo_cultivos_europa.joblib") // 1024, "KB")

try:
    from google.colab import files
    files.download("modelo_cultivos_europa.joblib")
    files.download("entrenamiento_eurostat_power.csv.gz")
except ImportError:
    pass  # fuera de Colab: los archivos quedan en esta carpeta

### 4. Qué devuelve el modelo
Para cada cultivo el modelo da la **cuota** (qué parte de la tierra se le dedica con ese clima y ese suelo) y el **rendimiento** en t/ha. Con el precio de venta de Eurostat se calcula la **ventaja** (lo que el cultivo rinde y se paga ahí frente a lo habitual en Europa) y la **puntuación = cuota × ventaja**, que es por lo que ordena el backend en `GET /recomendacion`.

Abajo, la recomendación para las cuatro fincas de la demo, con el clima de su último año y el pH que miden sus balizas (valores que calcula el backend).

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

# Producto de la tabla de precios de Eurostat (apri_ap_crpouta) de cada cultivo del modelo. Sin precio: maíz forrajero y lino.
PRECIOS = {"C1110": "01110000", "C1120": "01120000", "C1300": "01300000", "C1410": "01400000", "C1500": "01500000",
           "C1700": "01910000", "C2000": "01600000", "C1200": "01200000", "C1600": "01920000", "R1000": "05120000",
           "R2000": "02400000", "I1110": "02110000", "I1120": "02120000", "I1130": "02130000", "I3000": "02300000",
           "P1200": "02230000"}

def precios_eurostat():
    """{(producto, país): €/t} con la media de los 3 últimos años con dato, y {producto: mediana europea}."""
    d = _en_cache("precios.json", lambda: _json(f"{EUROSTAT}/apri_ap_crpouta?format=JSON&lang=EN&currency=EUR&sinceTimePeriod={ULTIMO_ANIO - 5}"))
    series = {}
    for fila, valor in _filas(d):
        if valor and valor > 0:
            por_tonelada = valor if fila["prod_veg"] == "02400000" else valor * 10   # €/100 kg -> €/t (la remolacha ya viene por tonelada)
            series.setdefault((fila["prod_veg"], fila["geo"]), {})[int(fila["time"])] = por_tonelada
    medias = {k: float(np.mean([v[a] for a in sorted(v)[-3:]])) for k, v in series.items()}
    europa = {p: float(np.median([m for (q, _), m in medias.items() if q == p])) for p in set(PRECIOS.values())
              if sum(q == p for q, _ in medias) >= 3}
    return medias, europa

precio_pais, precio_europa = precios_eurostat()

def recomendar(variables, pais="ES", top=8):
    marco = pd.DataFrame([[i] + [variables[v] for v in VARIABLES] for i in range(len(codigos))], columns=COLUMNAS)
    cuota = np.clip(m_cuota.predict(marco), 0, None)
    cuota = cuota / cuota.sum()
    relativo = np.clip(m_rendimiento.predict(marco[COLUMNAS_REND]), 0, None)
    filas = []
    for i, c in enumerate(codigos):
        producto = PRECIOS.get(c)
        europeo = precio_europa.get(producto)
        precio = precio_pais.get((producto, pais), europeo)       # sin dato del país: mediana europea
        ventaja = relativo[i] * (precio / europeo if precio and europeo else 1)
        rendimiento = relativo[i] * mediana[c]
        filas.append({"cultivo": CULTIVOS[c][1], "puntuación": round(100 * cuota[i] * ventaja, 1), "cuota %": round(100 * cuota[i], 1),
                      "ventaja": round(ventaja, 2), "rendimiento t/ha": round(rendimiento, 2),
                      "precio €/t": None if precio is None else round(precio),
                      "ingreso bruto €/ha": None if precio is None else round(rendimiento * precio)})
    return pd.DataFrame(filas).sort_values("puntuación", ascending=False).head(top).reset_index(drop=True)

# Clima del último año y pH de las balizas de cada finca de la demo (los calcula el backend con InfluxDB y Meteomatics).
FINCAS = {
    "c001 demo (Zaragoza, España)": ("ES", {"t_invierno": 8.6, "t_primavera": 15.9, "t_verano": 26.1, "t_otono": 17.5, "lluvia_invierno_mm": 24.3, "lluvia_primavera_mm": 73.7, "lluvia_verano_mm": 33.8, "lluvia_otono_mm": 39.2, "radiacion_primavera_mj_dia": 20.5, "radiacion_verano_mj_dia": 26.5, "hr_media": 54.7, "ph_suelo": 7.0}),
    "c002 Bajo Cinca (Fraga, España)": ("ES", {"t_invierno": 8.6, "t_primavera": 16.2, "t_verano": 27.0, "t_otono": 14.9, "lluvia_invierno_mm": 35.8, "lluvia_primavera_mm": 76.5, "lluvia_verano_mm": 49.7, "lluvia_otono_mm": 79.2, "radiacion_primavera_mj_dia": 20.8, "radiacion_verano_mj_dia": 26.3, "hr_media": 63.8, "ph_suelo": 8.0}),
    "c003 Slaney (Wexford, Irlanda)": ("IE", {"t_invierno": 7.2, "t_primavera": 10.2, "t_verano": 16.4, "t_otono": 10.9, "lluvia_invierno_mm": 444.6, "lluvia_primavera_mm": 174.9, "lluvia_verano_mm": 181.4, "lluvia_otono_mm": 215.0, "radiacion_primavera_mj_dia": 13.3, "radiacion_verano_mj_dia": 18.7, "hr_media": 82.2, "ph_suelo": 6.4}),
    "c004 Romerike (Nes, Noruega)": ("NO", {"t_invierno": -4.8, "t_primavera": 6.4, "t_verano": 15.9, "t_otono": 4.2, "lluvia_invierno_mm": 62.0, "lluvia_primavera_mm": 89.8, "lluvia_verano_mm": 214.4, "lluvia_otono_mm": 129.3, "radiacion_primavera_mj_dia": 13.4, "radiacion_verano_mj_dia": 18.0, "hr_media": 78.4, "ph_suelo": 6.2}),
}
for nombre, (pais, variables) in FINCAS.items():
    print("\n" + nombre)
    display(recomendar(variables, pais, top=6))

### 5. Pruébalo tú
Elige una finca y cambia el pH, la temperatura o la lluvia para ver cómo cambia la recomendación. En Colab los valores se cambian en el formulario de la derecha; después ejecuta la celda (▶).

In [ ]:
finca = "c002 Bajo Cinca (Fraga, España)"  #@param ["c001 demo (Zaragoza, España)", "c002 Bajo Cinca (Fraga, España)", "c003 Slaney (Wexford, Irlanda)", "c004 Romerike (Nes, Noruega)"]
ph_suelo = 8.0  #@param {type:"slider", min:4.5, max:8.5, step:0.1}
grados_de_mas = 0  #@param {type:"slider", min:-5, max:5, step:0.5}
lluvia_por_ciento = 100  #@param {type:"slider", min:25, max:300, step:25}

pais, variables = FINCAS[finca]
variables = dict(variables, ph_suelo=ph_suelo)
for v in VARIABLES:
    if v.startswith("t_"):
        variables[v] += grados_de_mas
    if v.startswith("lluvia_"):
        variables[v] *= lluvia_por_ciento / 100
print(f"{finca}: pH {ph_suelo}, temperatura {grados_de_mas:+} °C, lluvia al {lluvia_por_ciento} %")
display(recomendar(variables, pais, top=10))